In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    radius=1000,
    fletcher=False,
    c1=1e-4,
    tau=0.5,
    line_search_method="backtrack",
    line_search_cond="armijo",
    batch_size=1000
    # batch_size=None
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5158659815788269
epoch:  1, loss: 0.0746953934431076
epoch:  2, loss: 0.027985066175460815
epoch:  3, loss: 0.01376176718622446
epoch:  4, loss: 0.010617577470839024
epoch:  5, loss: 0.008920407854020596
epoch:  6, loss: 0.008799904957413673
epoch:  7, loss: 0.008128805086016655
epoch:  8, loss: 0.007326920982450247
epoch:  9, loss: 0.00706917978823185
epoch:  10, loss: 0.00646955706179142
epoch:  11, loss: 0.004709977190941572
epoch:  12, loss: 0.004592008423060179
epoch:  13, loss: 0.004093279596418142
epoch:  14, loss: 0.003839946584776044
epoch:  15, loss: 0.0030574495904147625
epoch:  16, loss: 0.0027332310564816
epoch:  17, loss: 0.002065743785351515
epoch:  18, loss: 0.001378138898871839
epoch:  19, loss: 0.0009788869647309184
epoch:  20, loss: 0.000784848933108151
epoch:  21, loss: 0.0006555455038323998
epoch:  22, loss: 0.0005052476190030575
epoch:  23, loss: 0.00046792207285761833
epoch:  24, loss: 0.0004434251750353724
epoch:  25, loss: 0.00039075262611731

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9989723563194275
Test metrics:  R2 = 0.9979797005653381


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0

/home/eugeniolr/Documents/Eugenio/torch_numopt/src/torch_numopt/utils.py:47: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:690.)
  U, S, Vt = torch.linalg.svd(mat)


, loss: 0.41828739643096924
epoch:  1, loss: 0.25735965371131897
epoch:  2, loss: 0.017878683283925056
epoch:  3, loss: 0.01558887492865324
epoch:  4, loss: 0.01272801123559475
epoch:  5, loss: 0.01053503155708313
epoch:  6, loss: 0.00962053146213293
epoch:  7, loss: 0.008398102596402168
epoch:  8, loss: 0.007940799929201603
epoch:  9, loss: 0.0076766167767345905
epoch:  10, loss: 0.007341945078223944
epoch:  11, loss: 0.006987135391682386
epoch:  12, loss: 0.006526441313326359
epoch:  13, loss: 0.006037082523107529
epoch:  14, loss: 0.0052804467268288136
epoch:  15, loss: 0.004541710484772921
epoch:  16, loss: 0.003969053737819195
epoch:  17, loss: 0.0035732495598495007
epoch:  18, loss: 0.0032438624184578657
epoch:  19, loss: 0.002969067543745041
epoch:  20, loss: 0.0027369505260139704
epoch:  21, loss: 0.002503670984879136
epoch:  22, loss: 0.0022973075974732637
epoch:  23, loss: 0.0020667219068855047
epoch:  24, loss: 0.0018431964563205838
epoch:  25, loss: 0.0016451373230665922
ep

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9922128915786743
Test metrics:  R2 = 0.989626944065094
